# V3: Evidence-Aware Context Engineering

V2 built a strong control plane: scoping, adaptive supervision, isolated workers, bounded
parallelism, and harness-owned completion. V3 strengthens the data plane. Research workers no
longer exchange large free-form notes; they produce normalized sources and atomic claims with
explicit provenance. A bounded context selector then gives the Writer only the evidence needed
for the current report.

```text
ResearchBrief
     |
     v
EvidenceResearchPlan
     | fan-out
     v
Isolated Evidence Workers
     | ResearchPacketDraft
     v
Normalizer + EvidenceStore
     | selected claims and sources
     v
Writer -> Citation Validator -> Report
```

# 0. Setup

## 0.1 Imports

In [1]:
import asyncio
import hashlib
import json
import os
import platform
import re
import sys
from datetime import UTC, datetime
from pathlib import Path
from typing import Any
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import AsyncOpenAI, OpenAI
from pydantic import BaseModel, ConfigDict, Field

from agentic_deepresearch.schemas import (
    CitationValidationResult,
    EvidenceClaim,
    EvidenceResearchPlan,
    EvidenceResearchTask,
    EvidenceStore,
    EvidenceWriterOutput,
    ResearchBrief,
    ResearchPacket,
    ResearchPacketDraft,
    SourceRecord,
    WriterContext,
)
from agentic_deepresearch.tools import (
    arxiv_search_tool,
    arxiv_tool_def,
    tavily_search_tool,
    tavily_tool_def,
)

## 0.2 Runtime Configuration

In [2]:
EXPECTED_PYTHON_VERSION = (3, 12)

env_file = find_dotenv(usecwd=True)
if not env_file:
    raise RuntimeError("Could not locate the project .env file.")

ENV_PATH = Path(env_file).resolve()
PROJECT_ROOT = ENV_PATH.parent

if sys.version_info[:2] != EXPECTED_PYTHON_VERSION:
    raise RuntimeError("Python 3.12 is required. Start Jupyter with `uv run jupyter lab`.")

load_dotenv(dotenv_path=ENV_PATH, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is not configured in .env.")
if not MODEL_NAME:
    raise RuntimeError("MODEL_NAME is not configured in .env.")

client = OpenAI(api_key=OPENAI_API_KEY)
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

print(f"[SETUP] Python: {platform.python_version()}")
print(f"[SETUP] Project directory: {PROJECT_ROOT}")
print(f"[SETUP] Model: {MODEL_NAME}")
print(f"[SETUP] Tavily configured: {bool(os.getenv('TAVILY_API_KEY'))}")

[SETUP] Python: 3.12.12
[SETUP] Project directory: /Users/daboluo/MyWorkSpace/GitHub/agentic-deepresearch
[SETUP] Model: gpt-5-nano
[SETUP] Tavily configured: True


In [3]:
def show(title: str, content: str) -> None:
    display(Markdown(f"## {title}\n\n{content}"))

# 1. Evidence Contracts

A model is good at extracting semantic relationships but should not own database identity. The
worker therefore returns `ResearchPacketDraft` with URLs. Ordinary Python canonicalizes those
URLs, creates stable source and claim identifiers, and validates every internal reference. The
shared Pydantic contracts live in `agentic_deepresearch.schemas`.

## 1.1 Deterministic URL and Identifier Normalization

In [4]:
TRACKING_QUERY_PREFIXES = ("utm_", "ref", "source")


def canonicalize_url(url: str) -> str:
    """Normalize a source URL for deterministic deduplication."""

    raw_url = url.strip()
    parts = urlsplit(raw_url)
    if parts.scheme not in {"http", "https"} or not parts.netloc:
        raise ValueError(f"Source URL must be absolute HTTP(S): {url!r}")

    scheme = "https"
    host = parts.netloc.lower()
    host = host.removeprefix("www.")

    path = re.sub(r"/{2,}", "/", parts.path).rstrip("/") or "/"
    if host in {"arxiv.org", "export.arxiv.org"}:
        arxiv_match = re.search(r"/(?:abs|pdf)/(\d+\.\d+)(?:v\d+)?(?:\.pdf)?$", path)
        if arxiv_match:
            host = "arxiv.org"
            path = f"/abs/{arxiv_match.group(1)}"

    query_items = [
        (key, value)
        for key, value in parse_qsl(parts.query, keep_blank_values=True)
        if not key.lower().startswith(TRACKING_QUERY_PREFIXES)
    ]
    query = urlencode(sorted(query_items))
    return urlunsplit((scheme, host, path, query, ""))


def stable_id(prefix: str, value: str) -> str:
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()[:12]
    return f"{prefix}_{digest}"


print(canonicalize_url("http://www.arxiv.org/pdf/2401.12345v2.pdf?utm_source=test"))

https://arxiv.org/abs/2401.12345


## 1.2 Compile a Worker Draft into Durable Evidence

In [5]:
class CompiledResearchPacket(BaseModel):
    model_config = ConfigDict(extra="forbid")

    packet: ResearchPacket
    sources: list[SourceRecord]
    claims: list[EvidenceClaim]


class MergeStats(BaseModel):
    model_config = ConfigDict(extra="forbid")

    added_sources: int = Field(ge=0)
    reused_sources: int = Field(ge=0)
    added_claims: int = Field(ge=0)
    reused_claims: int = Field(ge=0)

In [6]:
def compile_research_packet(
    draft: ResearchPacketDraft,
    worker_id: int,
    retrieved_at: str | None = None,
) -> CompiledResearchPacket:
    """Assign stable IDs and reject claims that reference undeclared sources."""

    retrieval_time = retrieved_at or datetime.now(UTC).isoformat()
    source_by_url: dict[str, SourceRecord] = {}

    for candidate in draft.sources:
        canonical_url = canonicalize_url(candidate.url)
        source_id = stable_id("src", canonical_url)
        source_by_url[canonical_url] = SourceRecord(
            id=source_id,
            title=candidate.title,
            url=candidate.url,
            canonical_url=canonical_url,
            source_type=candidate.source_type,
            authors=candidate.authors,
            published_date=candidate.published_date,
            retrieved_at=retrieval_time,
        )

    claims: list[EvidenceClaim] = []
    for candidate in draft.claims:
        canonical_urls = [canonicalize_url(url) for url in candidate.source_urls]
        missing_urls = sorted(set(canonical_urls) - set(source_by_url))
        if missing_urls:
            raise ValueError(
                "A claim references URLs that are absent from the packet sources: "
                f"{missing_urls!r}"
            )

        source_ids = sorted({source_by_url[url].id for url in canonical_urls})
        normalized_claim = re.sub(r"\s+", " ", candidate.claim.strip().lower())
        claim_fingerprint = normalized_claim
        claims.append(
            EvidenceClaim(
                id=stable_id("clm", claim_fingerprint),
                claim=candidate.claim,
                source_ids=source_ids,
                evidence_excerpt=candidate.evidence_excerpt,
                confidence=candidate.confidence,
                limitations=candidate.limitations,
            )
        )

    sources = sorted(source_by_url.values(), key=lambda source: source.id)
    packet = ResearchPacket(
        worker_id=worker_id,
        task=draft.task,
        summary=draft.summary,
        source_ids=[source.id for source in sources],
        claim_ids=[claim.id for claim in claims],
        unresolved_questions=draft.unresolved_questions,
    )
    return CompiledResearchPacket(packet=packet, sources=sources, claims=claims)

# 2. Evidence Store

`EvidenceStore` is run-scoped working memory. It is not a vector database or long-term user
memory. Its job is to merge concurrent worker packets, deduplicate sources, preserve claim
provenance, and expose unresolved evidence gaps.

In [7]:
def merge_research_packet(
    store: EvidenceStore,
    compiled: CompiledResearchPacket,
) -> tuple[EvidenceStore, MergeStats]:
    """Return a new validated store and deterministic merge statistics."""

    updated = store.model_copy(deep=True)
    added_sources = 0
    reused_sources = 0
    added_claims = 0
    reused_claims = 0

    for source in compiled.sources:
        if source.id in updated.sources:
            reused_sources += 1
        else:
            updated.sources[source.id] = source
            added_sources += 1

    confidence_rank = {"low": 0, "medium": 1, "high": 2}
    for claim in compiled.claims:
        existing_claim = updated.claims.get(claim.id)
        if existing_claim is not None:
            reused_claims += 1
            merged_confidence = max(
                [existing_claim.confidence, claim.confidence],
                key=lambda value: confidence_rank[value],
            )
            updated.claims[claim.id] = existing_claim.model_copy(
                update={
                    "source_ids": sorted(
                        set(existing_claim.source_ids) | set(claim.source_ids)
                    ),
                    "confidence": merged_confidence,
                    "limitations": list(
                        dict.fromkeys([*existing_claim.limitations, *claim.limitations])
                    ),
                }
            )
        else:
            updated.claims[claim.id] = claim
            added_claims += 1

    updated.packets.append(compiled.packet)
    known_questions = set(updated.unresolved_questions)
    for question in compiled.packet.unresolved_questions:
        normalized = question.strip()
        if normalized and normalized not in known_questions:
            updated.unresolved_questions.append(normalized)
            known_questions.add(normalized)

    validated = EvidenceStore.model_validate(updated.model_dump())
    stats = MergeStats(
        added_sources=added_sources,
        reused_sources=reused_sources,
        added_claims=added_claims,
        reused_claims=reused_claims,
    )
    return validated, stats


def summarize_evidence_store(store: EvidenceStore) -> dict[str, Any]:
    source_types: dict[str, int] = {}
    for source in store.sources.values():
        source_types[source.source_type] = source_types.get(source.source_type, 0) + 1
    return {
        "packets": len(store.packets),
        "sources": len(store.sources),
        "claims": len(store.claims),
        "source_types": source_types,
        "unresolved_questions": len(store.unresolved_questions),
    }

# 3. Bounded Evidence Research

## 3.1 Evidence Planner

In [8]:
EVIDENCE_PLANNER_INSTRUCTIONS = """
You are the evidence planner for a deep research workflow.

Decompose the ResearchBrief into independent evidence lanes that can run concurrently.
Each task must ask a focused question and state observable evidence requirements.
Use non-overlapping tasks and cover the brief's major comparison dimensions.
Do not write the report, choose final conclusions, or include conversational text.
"""


def evidence_planner_agent(
    research_brief: ResearchBrief,
    max_tasks: int = 3,
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> EvidenceResearchPlan:
    if max_tasks < 1:
        raise ValueError("max_tasks must be positive.")
    if verbose:
        print("[EVIDENCE_PLANNER] Model call #1")

    response = client.responses.parse(
        model=model,
        instructions=(
            EVIDENCE_PLANNER_INSTRUCTIONS
            + f"\nReturn between 1 and {max_tasks} tasks with consecutive one-based IDs."
        ),
        input=json.dumps(research_brief.model_dump(), ensure_ascii=False),
        text_format=EvidenceResearchPlan,
    )
    plan = response.output_parsed
    if plan is None:
        raise RuntimeError("Evidence planner returned no structured plan.")
    if len(plan.tasks) > max_tasks:
        raise RuntimeError("Evidence planner exceeded the task budget.")
    expected_ids = list(range(1, len(plan.tasks) + 1))
    if [task.id for task in plan.tasks] != expected_ids:
        raise RuntimeError("Evidence task IDs must be consecutive and one-based.")

    if verbose:
        print(f"[EVIDENCE_PLANNER] Planned tasks: {len(plan.tasks)}")
    return plan

## 3.2 Isolated Evidence Worker

In [9]:
TOOLS = [arxiv_tool_def, tavily_tool_def]
TOOL_FUNCTIONS = {
    "arxiv_search_tool": arxiv_search_tool,
    "tavily_search_tool": tavily_search_tool,
}


def execute_tool(name: str, arguments: dict[str, Any]) -> list[dict[str, Any]]:
    tool = TOOL_FUNCTIONS.get(name)
    if tool is None:
        raise ValueError(f"Unknown tool: {name!r}")
    return tool(**arguments)


EVIDENCE_RESEARCH_INSTRUCTIONS = """
You are an isolated evidence research worker.

Investigate only the assigned question. Use arXiv for academic literature and Tavily for
credible web or industry sources. Prefer primary sources. Gather enough evidence to satisfy
the listed requirements, then stop calling tools. Do not assume access to other workers.
Do not invent source metadata, URLs, claims, or publication status.
"""


PACKET_SYNTHESIS_INSTRUCTIONS = """
Convert the collected tool evidence into one ResearchPacketDraft.

Rules:
- Include only sources that were present in tool results.
- Preserve the exact source URL returned by a tool.
- Classify arXiv-only work as preprint unless the evidence proves peer review.
- Make every claim atomic and map it to one or more URLs in the sources list.
- Paraphrase supporting evidence; do not fabricate quotations.
- Record material uncertainty in limitations and unresolved_questions.
- Return only the structured result.
"""


PACKET_REPAIR_INSTRUCTIONS = """
Repair an invalid ResearchPacketDraft using only the collected tool evidence. Every URL named
by a claim must have a matching entry in sources. Add an omitted source only when its metadata
exists in the tool evidence; otherwise remove that URL from the claim or remove the unsupported
claim. Preserve exact tool-returned URLs and return only the corrected structured result.
"""

In [10]:
async def evidence_research_worker(
    research_brief: ResearchBrief,
    task: EvidenceResearchTask,
    model: str = MODEL_NAME,
    max_tool_rounds: int = 4,
    verbose: bool = True,
) -> CompiledResearchPacket:
    """Run one isolated tool loop, then compile its structured evidence packet."""

    if max_tool_rounds < 1:
        raise ValueError("max_tool_rounds must be positive.")

    prefix = f"[EVIDENCE_WORKER:{task.id}]"
    request = {
        "research_brief": research_brief.model_dump(),
        "task": task.model_dump(),
    }
    input_items: list[Any] = [
        {"role": "user", "content": json.dumps(request, ensure_ascii=False)}
    ]
    tool_rounds = 0
    tool_calls_total = 0

    if verbose:
        print(f"{prefix} Starting isolated evidence task.")

    while True:
        budget_exhausted = tool_rounds >= max_tool_rounds
        response = await async_client.responses.create(
            model=model,
            instructions=EVIDENCE_RESEARCH_INSTRUCTIONS,
            input=input_items,
            tools=TOOLS,
            tool_choice="none" if budget_exhausted else "auto",
            parallel_tool_calls=False,
        )
        input_items.extend(response.output)
        tool_calls = [item for item in response.output if item.type == "function_call"]
        if not tool_calls:
            break

        for tool_call in tool_calls:
            tool_calls_total += 1
            try:
                arguments = json.loads(tool_call.arguments)
                result = await asyncio.to_thread(
                    execute_tool, tool_call.name, arguments
                )
                errors = [
                    str(item["error"])
                    for item in result
                    if isinstance(item, dict) and item.get("error")
                ]
                tool_output = (
                    {"ok": False, "error": "; ".join(errors)}
                    if errors
                    else {"ok": True, "result": result}
                )
                result_summary = f"items={len(result)}, errors={len(errors)}"
            except Exception as exc:  # noqa: BLE001
                tool_output = {"ok": False, "error": str(exc)}
                result_summary = f"error={type(exc).__name__}"

            input_items.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(tool_output, ensure_ascii=False),
                }
            )
            if verbose:
                print(
                    f"{prefix} Tool call #{tool_calls_total}: "
                    f"{tool_call.name} ({result_summary})"
                )
        tool_rounds += 1

    synthesis = await async_client.responses.parse(
        model=model,
        instructions=PACKET_SYNTHESIS_INSTRUCTIONS,
        input=[
            *input_items,
            {
                "role": "user",
                "content": (
                    "Produce the final ResearchPacketDraft for the assigned task using only "
                    "the evidence collected above."
                ),
            },
        ],
        text_format=ResearchPacketDraft,
    )
    draft = synthesis.output_parsed
    if draft is None:
        raise RuntimeError(f"Worker {task.id} returned no structured evidence packet.")
    draft.task = task.question
    try:
        compiled = compile_research_packet(draft=draft, worker_id=task.id)
    except ValueError as exc:
        if verbose:
            print(f"{prefix} Packet repair call: {exc}")
        repair = await async_client.responses.parse(
            model=model,
            instructions=PACKET_REPAIR_INSTRUCTIONS,
            input=[
                *input_items,
                {
                    "role": "user",
                    "content": json.dumps(
                        {
                            "validation_error": str(exc),
                            "invalid_packet": draft.model_dump(),
                        },
                        ensure_ascii=False,
                    ),
                },
            ],
            text_format=ResearchPacketDraft,
        )
        repaired_draft = repair.output_parsed
        if repaired_draft is None:
            raise RuntimeError(f"Worker {task.id} packet repair returned no output.")
        repaired_draft.task = task.question
        compiled = compile_research_packet(
            draft=repaired_draft, worker_id=task.id
        )

    if verbose:
        print(
            f"{prefix} Completed: sources={len(compiled.sources)}, "
            f"claims={len(compiled.claims)}, tool_calls={tool_calls_total}"
        )
    return compiled

## 3.3 Bounded Parallel Fan-Out and Fan-In

In [11]:
async def run_evidence_workers(
    research_brief: ResearchBrief,
    plan: EvidenceResearchPlan,
    max_parallel_workers: int = 2,
    max_tool_rounds: int = 4,
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> list[CompiledResearchPacket]:
    if max_parallel_workers < 1:
        raise ValueError("max_parallel_workers must be positive.")

    semaphore = asyncio.Semaphore(max_parallel_workers)

    async def run_one(task: EvidenceResearchTask) -> CompiledResearchPacket:
        async with semaphore:
            return await evidence_research_worker(
                research_brief=research_brief,
                task=task,
                model=model,
                max_tool_rounds=max_tool_rounds,
                verbose=verbose,
            )

    results = await asyncio.gather(
        *(run_one(task) for task in plan.tasks),
        return_exceptions=True,
    )
    failures = [result for result in results if isinstance(result, BaseException)]
    if failures:
        failure_messages = [f"{type(exc).__name__}: {exc}" for exc in failures]
        raise RuntimeError(f"Evidence worker batch failed: {failure_messages!r}")
    return [result for result in results if isinstance(result, CompiledResearchPacket)]


def merge_evidence_batch(
    packets: list[CompiledResearchPacket],
) -> tuple[EvidenceStore, list[MergeStats]]:
    store = EvidenceStore()
    merge_stats: list[MergeStats] = []
    for compiled in sorted(packets, key=lambda item: item.packet.worker_id):
        store, stats = merge_research_packet(store, compiled)
        merge_stats.append(stats)
    return store, merge_stats

# 4. Context Selection

The Writer should not receive every raw tool result or every worker transcript. The selector
scores atomic claims against the scoped brief, applies a hard claim budget, and includes only
the sources referenced by selected claims. This makes context growth explicit and testable.

In [12]:
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for",
    "from", "how", "in", "is", "of", "on", "or", "that", "the",
    "their", "to", "what", "which", "with",
}
CONFIDENCE_WEIGHT = {"high": 1.0, "medium": 0.5, "low": 0.0}


def semantic_tokens(text: str) -> set[str]:
    return {
        token
        for token in re.findall(r"[a-z0-9]+", text.lower())
        if len(token) > 2 and token not in STOP_WORDS
    }


def score_claim(claim: EvidenceClaim, query_tokens: set[str]) -> float:
    claim_tokens = semantic_tokens(
        " ".join([claim.claim, claim.evidence_excerpt, *claim.limitations])
    )
    overlap = len(claim_tokens & query_tokens) / max(1, len(query_tokens))
    source_bonus = min(len(claim.source_ids), 3) * 0.1
    return overlap + CONFIDENCE_WEIGHT[claim.confidence] + source_bonus


def select_writer_context(
    research_brief: ResearchBrief,
    store: EvidenceStore,
    max_claims: int = 16,
    max_unresolved_questions: int = 5,
) -> WriterContext:
    if max_claims < 1:
        raise ValueError("max_claims must be positive.")

    query_text = " ".join(
        [
            research_brief.research_question,
            research_brief.objective,
            *research_brief.scope_inclusions,
            *research_brief.success_criteria,
        ]
    )
    query_tokens = semantic_tokens(query_text)
    ranked_claims = sorted(
        store.claims.values(),
        key=lambda claim: (score_claim(claim, query_tokens), claim.id),
        reverse=True,
    )[:max_claims]
    selected_source_ids = {
        source_id for claim in ranked_claims for source_id in claim.source_ids
    }
    selected_sources = [
        store.sources[source_id] for source_id in sorted(selected_source_ids)
    ]
    return WriterContext(
        research_brief=research_brief,
        claims=ranked_claims,
        sources=selected_sources,
        unresolved_questions=store.unresolved_questions[:max_unresolved_questions],
    )


CITATION_REPAIR_INSTRUCTIONS = """
You repair citations in an evidence-aware research report. Rewrite the report so every body
citation uses an exact source identifier supplied in the context. Do not invent identifiers,
sources, URLs, or claims. Preserve the useful analysis, remove or soften unsupported statements,
and return only the structured writer output. A deterministic Python step will rebuild the final
References section, so concentrate on correct inline citations and evidence fidelity.
"""


def report_body(markdown: str) -> str:
    heading_match = REFERENCE_HEADING_PATTERN.search(markdown)
    return markdown[:heading_match.start()].rstrip() if heading_match else markdown.rstrip()


def remap_model_source_ids(
    markdown: str, store: EvidenceStore
) -> tuple[str, dict[str, str]]:
    heading_match = REFERENCE_HEADING_PATTERN.search(markdown)
    if not heading_match:
        return markdown, {}

    references_text = markdown[heading_match.end():]
    canonical_to_id = {
        source.canonical_url: source.id for source in store.sources.values()
    }
    source_id_map: dict[str, str] = {}
    for line in references_text.splitlines():
        provisional_ids = SOURCE_CITATION_PATTERN.findall(line)
        urls = re.findall(r"https?://[^\s)>]+", line)
        if not provisional_ids or not urls:
            continue
        for url in urls:
            canonical_url = canonicalize_url(url.rstrip(".,;"))
            stable_source_id = canonical_to_id.get(canonical_url)
            if stable_source_id is not None:
                for provisional_id in provisional_ids:
                    source_id_map[provisional_id] = stable_source_id
                break

    remapped_markdown = markdown
    for provisional_id, stable_source_id in source_id_map.items():
        remapped_markdown = remapped_markdown.replace(
            provisional_id, stable_source_id
        )
    return remapped_markdown, source_id_map


def fail_closed_citation_filter(
    markdown: str, store: EvidenceStore
) -> tuple[str, list[str], int]:
    heading_match = REFERENCE_HEADING_PATTERN.search(markdown)
    body = markdown[:heading_match.start()] if heading_match else markdown
    references_suffix = markdown[heading_match.start():] if heading_match else ""
    known_source_ids = set(store.sources)
    removed_source_ids: set[str] = set()
    dropped_lines = 0
    citation_group_pattern = re.compile(r"\[[^\]]*src_[0-9a-f]{12}[^\]]*\]")

    filtered_lines: list[str] = []
    for line in body.splitlines():
        original_ids = SOURCE_CITATION_PATTERN.findall(line)
        if not original_ids:
            filtered_lines.append(line)
            continue

        def filter_group(match: re.Match[str]) -> str:
            group_ids = SOURCE_CITATION_PATTERN.findall(match.group(0))
            valid_ids = list(dict.fromkeys(
                source_id for source_id in group_ids if source_id in known_source_ids
            ))
            removed_source_ids.update(
                source_id for source_id in group_ids if source_id not in known_source_ids
            )
            return f"[{', '.join(valid_ids)}]" if valid_ids else ""

        filtered_line = citation_group_pattern.sub(filter_group, line)
        if not SOURCE_CITATION_PATTERN.findall(filtered_line):
            dropped_lines += 1
            continue
        filtered_lines.append(filtered_line)

    filtered_body = "\n".join(filtered_lines).rstrip()
    filtered_markdown = (
        filtered_body + "\n\n" + references_suffix.lstrip()
        if references_suffix
        else filtered_body
    )
    return filtered_markdown, sorted(removed_source_ids), dropped_lines


def canonicalize_writer_references(
    writer_output: EvidenceWriterOutput,
    context: WriterContext,
    store: EvidenceStore,
) -> EvidenceWriterOutput:
    remapped_markdown, _ = remap_model_source_ids(writer_output.markdown, store)
    filtered_markdown, _, _ = fail_closed_citation_filter(remapped_markdown, store)
    body = report_body(filtered_markdown)
    body_source_ids = sorted(set(SOURCE_CITATION_PATTERN.findall(body)))
    unknown_source_ids = sorted(set(body_source_ids) - set(store.sources))
    if unknown_source_ids:
        raise ValueError(f"Cannot canonicalize unknown source IDs: {unknown_source_ids!r}")
    if not body_source_ids:
        raise ValueError("Cannot canonicalize a report without body citations.")

    reference_lines = [
        f"- [{source_id}] {store.sources[source_id].title} — "
        f"{store.sources[source_id].url}"
        for source_id in body_source_ids
    ]
    context_claim_ids = {claim.id for claim in context.claims}
    used_claim_ids = sorted(set(writer_output.used_claim_ids) & context_claim_ids)
    markdown = body + "\n\n## References\n\n" + "\n".join(reference_lines)
    return EvidenceWriterOutput(
        markdown=markdown,
        used_claim_ids=used_claim_ids,
        used_source_ids=body_source_ids,
    )


def finalize_writer_output(
    writer_output: EvidenceWriterOutput,
    context: WriterContext,
    store: EvidenceStore,
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> EvidenceWriterOutput:
    remapped_markdown, source_id_map = remap_model_source_ids(
        writer_output.markdown, store
    )
    if source_id_map and verbose:
        print(f"[EVIDENCE_WRITER] Remapped source IDs from URLs: {source_id_map}")
    filtered_markdown, removed_source_ids, dropped_lines = (
        fail_closed_citation_filter(remapped_markdown, store)
    )
    if removed_source_ids and verbose:
        print(
            "[EVIDENCE_WRITER] Fail-closed citation filter: "
            f"removed={removed_source_ids}, dropped_lines={dropped_lines}"
        )
    writer_output = writer_output.model_copy(
        update={"markdown": filtered_markdown}
    )
    body = report_body(writer_output.markdown)
    body_source_ids = set(SOURCE_CITATION_PATTERN.findall(body))
    unknown_source_ids = sorted(body_source_ids - set(store.sources))

    if not body_source_ids:
        if verbose:
            print(
                "[EVIDENCE_WRITER] Citation repair call: "
                f"unknown={unknown_source_ids}, body_citations={len(body_source_ids)}"
            )
        payload = {
            "report": writer_output.model_dump(),
            "allowed_claims": [claim.model_dump() for claim in context.claims],
            "allowed_sources": [source.model_dump() for source in context.sources],
        }
        response = client.responses.parse(
            model=model,
            instructions=CITATION_REPAIR_INSTRUCTIONS,
            input=json.dumps(payload, ensure_ascii=False),
            text_format=EvidenceWriterOutput,
        )
        repaired_output = response.output_parsed
        if repaired_output is None:
            raise RuntimeError("Citation repair returned no structured output.")
        remapped_markdown, repaired_id_map = remap_model_source_ids(
            repaired_output.markdown, store
        )
        if repaired_id_map and verbose:
            print(
                "[EVIDENCE_WRITER] Remapped repaired source IDs from URLs: "
                f"{repaired_id_map}"
            )
        filtered_markdown, repaired_removed_ids, repaired_dropped_lines = (
            fail_closed_citation_filter(remapped_markdown, store)
        )
        if repaired_removed_ids and verbose:
            print(
                "[EVIDENCE_WRITER] Repaired output fail-closed filter: "
                f"removed={repaired_removed_ids}, "
                f"dropped_lines={repaired_dropped_lines}"
            )
        writer_output = repaired_output.model_copy(
            update={"markdown": filtered_markdown}
        )

    finalized_output = canonicalize_writer_references(writer_output, context, store)
    if verbose:
        print(
            "[EVIDENCE_WRITER] References canonicalized: "
            f"{len(finalized_output.used_source_ids)} cited sources"
        )
    return finalized_output

# 5. Evidence-Aware Writer and Citation Validation

In [13]:
EVIDENCE_WRITER_INSTRUCTIONS = """
You are the Writer Agent for an evidence-aware deep research system.

Write the complete Markdown report requested by the ResearchBrief using only the supplied
claims and sources. Cite factual statements with source identifiers in square brackets, such
as [src_0123456789ab]. Include ## Abstract, the main analysis, ## Limitations, and
## References. In References, use exactly this format for each cited source:
- [src_0123456789ab] Source title — https://example.com/source

Do not cite a source that is absent from the context. Do not convert unresolved questions into
facts. Return only the structured writer output. Do not add offers, follow-up questions, or
conversational framing.
"""


def evidence_writer_agent(
    context: WriterContext,
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> EvidenceWriterOutput:
    if not context.claims or not context.sources:
        raise ValueError("Writer context requires at least one claim and source.")
    if verbose:
        print(
            f"[EVIDENCE_WRITER] Model call #1: claims={len(context.claims)}, "
            f"sources={len(context.sources)}"
        )

    response = client.responses.parse(
        model=model,
        instructions=EVIDENCE_WRITER_INSTRUCTIONS,
        input=json.dumps(context.model_dump(), ensure_ascii=False),
        text_format=EvidenceWriterOutput,
    )
    output = response.output_parsed
    if output is None:
        raise RuntimeError("Evidence Writer returned no structured output.")
    return output

In [14]:
SOURCE_CITATION_PATTERN = re.compile(r"src_[0-9a-f]{12}")
REFERENCE_HEADING_PATTERN = re.compile(r"^## References\s*$", re.MULTILINE)


def validate_report_citations(
    writer_output: EvidenceWriterOutput,
    store: EvidenceStore,
) -> CitationValidationResult:
    report = writer_output.markdown
    cited_source_ids = sorted(set(SOURCE_CITATION_PATTERN.findall(report)))
    known_source_ids = set(store.sources)
    unknown_source_ids = sorted(set(cited_source_ids) - known_source_ids)

    heading_match = REFERENCE_HEADING_PATTERN.search(report)
    references_text = report[heading_match.end():] if heading_match else ""
    reference_ids = set(SOURCE_CITATION_PATTERN.findall(references_text))
    body_text = report[:heading_match.start()] if heading_match else report
    body_citation_ids = set(SOURCE_CITATION_PATTERN.findall(body_text))
    missing_reference_ids = sorted(body_citation_ids - reference_ids)
    invalid_reference_urls = sorted(
        source_id
        for source_id in body_citation_ids & reference_ids & known_source_ids
        if store.sources[source_id].url not in references_text
        and store.sources[source_id].canonical_url not in references_text
    )

    declared_source_ids = set(writer_output.used_source_ids)
    undeclared_cited_source_ids = sorted(body_citation_ids - declared_source_ids)
    unused_declared_source_ids = sorted(declared_source_ids - body_citation_ids)
    unknown_claim_ids = sorted(set(writer_output.used_claim_ids) - set(store.claims))

    issues: list[str] = []
    if not heading_match:
        issues.append("The report is missing a ## References section.")
    if not body_citation_ids:
        issues.append("The report body contains no source citations.")
    if unknown_source_ids:
        issues.append(f"Unknown source IDs: {unknown_source_ids!r}")
    if missing_reference_ids:
        issues.append(f"Citations missing from References: {missing_reference_ids!r}")
    if invalid_reference_urls:
        issues.append(
            f"Reference entries missing their stored URLs: {invalid_reference_urls!r}"
        )
    if undeclared_cited_source_ids:
        issues.append(
            f"Body citations absent from used_source_ids: {undeclared_cited_source_ids!r}"
        )
    if unused_declared_source_ids:
        issues.append(
            f"Declared source IDs not cited in the body: {unused_declared_source_ids!r}"
        )
    if unknown_claim_ids:
        issues.append(f"Unknown claim IDs: {unknown_claim_ids!r}")

    return CitationValidationResult(
        valid=not issues,
        cited_source_ids=cited_source_ids,
        unknown_source_ids=unknown_source_ids,
        undeclared_cited_source_ids=undeclared_cited_source_ids,
        missing_reference_ids=missing_reference_ids,
        unused_declared_source_ids=unused_declared_source_ids,
        issues=issues,
    )

## 5.1 Optional Semantic Citation Assessment

Python can prove that an identifier and URL exist, but it cannot prove that a source really
supports the surrounding sentence. This optional evaluator handles that semantic question.

In [15]:
class SemanticCitationAssessment(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    supported: bool
    unsupported_statements: list[str]
    overstated_claims: list[str]
    missing_limitations: list[str]
    summary: str = Field(min_length=1)


SEMANTIC_CITATION_INSTRUCTIONS = """
Evaluate whether the report's cited factual statements are supported by the supplied atomic
claims and evidence excerpts. Flag unsupported statements, overstatement, and omitted material
limitations. Do not judge writing style. Return only the structured assessment.
"""


def semantic_citation_agent(
    writer_output: EvidenceWriterOutput,
    context: WriterContext,
    model: str = MODEL_NAME,
) -> SemanticCitationAssessment:
    payload = {
        "report": writer_output.markdown,
        "claims": [claim.model_dump() for claim in context.claims],
        "sources": [source.model_dump() for source in context.sources],
    }
    response = client.responses.parse(
        model=model,
        instructions=SEMANTIC_CITATION_INSTRUCTIONS,
        input=json.dumps(payload, ensure_ascii=False),
        text_format=SemanticCitationAssessment,
    )
    assessment = response.output_parsed
    if assessment is None:
        raise RuntimeError("Semantic citation evaluator returned no assessment.")
    return assessment

# 6. V3 End-to-End Controller

In [16]:
class V3RunResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    plan: EvidenceResearchPlan
    evidence_store: EvidenceStore
    writer_context: WriterContext
    writer_output: EvidenceWriterOutput
    citation_validation: CitationValidationResult
    semantic_assessment: SemanticCitationAssessment | None = None
    report_path: str

In [17]:
async def run_v3_evidence_research(
    research_brief: ResearchBrief,
    max_tasks: int = 3,
    max_parallel_workers: int = 2,
    max_tool_rounds: int = 4,
    max_writer_claims: int = 16,
    run_semantic_evaluator: bool = False,
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> V3RunResult:
    """Plan, research, normalize, select context, write, and validate one report."""

    plan = await asyncio.to_thread(
        evidence_planner_agent, research_brief, max_tasks, model, verbose
    )
    packets = await run_evidence_workers(
        research_brief=research_brief,
        plan=plan,
        max_parallel_workers=max_parallel_workers,
        max_tool_rounds=max_tool_rounds,
        model=model,
        verbose=verbose,
    )
    store, merge_stats = merge_evidence_batch(packets)
    if verbose:
        print(f"[V3_CONTROLLER] Evidence store: {summarize_evidence_store(store)}")
        print(
            "[V3_CONTROLLER] Reused sources: "
            f"{sum(item.reused_sources for item in merge_stats)}"
        )

    context = select_writer_context(
        research_brief=research_brief,
        store=store,
        max_claims=max_writer_claims,
    )
    writer_output = await asyncio.to_thread(
        evidence_writer_agent, context, model, verbose
    )
    writer_output = await asyncio.to_thread(
        finalize_writer_output, writer_output, context, store, model, verbose
    )
    validation = validate_report_citations(writer_output, store)
    if not validation.valid:
        raise RuntimeError(f"Citation validation failed: {validation.issues!r}")

    semantic_assessment = None
    if run_semantic_evaluator:
        semantic_assessment = await asyncio.to_thread(
            semantic_citation_agent, writer_output, context, model
        )
        if not semantic_assessment.supported:
            raise RuntimeError(
                f"Semantic citation assessment failed: {semantic_assessment.model_dump()}"
            )

    if verbose:
        print("[V3_CONTROLLER] Report retained in notebook memory.")

    return V3RunResult(
        plan=plan,
        evidence_store=store,
        writer_context=context,
        writer_output=writer_output,
        citation_validation=validation,
        semantic_assessment=semantic_assessment,
    )

# 7. Deterministic Evidence Regression Check

In [18]:
def verify_v3_evidence_pipeline() -> dict[str, Any]:
    source_url = "https://arxiv.org/abs/2401.12345"
    duplicate_url = "http://www.arxiv.org/pdf/2401.12345v2.pdf?utm_source=demo"
    base_draft = {
        "task": "Study verification loops for tool-using agents.",
        "summary": "Verification can detect some execution failures.",
        "sources": [
            {
                "title": "Synthetic Verification Study",
                "url": source_url,
                "source_type": "preprint",
                "authors": ["A. Researcher"],
                "published_date": "2024-01-01",
            }
        ],
        "claims": [
            {
                "claim": "Verification loops can detect failed tool executions.",
                "source_urls": [source_url],
                "evidence_excerpt": "The study evaluates a post-action verification step.",
                "confidence": "medium",
                "limitations": ["The source is a preprint."],
            }
        ],
        "unresolved_questions": ["How does verification affect latency?"],
    }
    first = compile_research_packet(
        ResearchPacketDraft.model_validate(base_draft),
        worker_id=1,
        retrieved_at="2026-01-01T00:00:00+00:00",
    )

    second_payload = json.loads(json.dumps(base_draft))
    second_payload["sources"][0]["url"] = duplicate_url
    second_payload["claims"][0]["source_urls"] = [duplicate_url]
    second = compile_research_packet(
        ResearchPacketDraft.model_validate(second_payload),
        worker_id=2,
        retrieved_at="2026-01-01T00:01:00+00:00",
    )

    store, first_stats = merge_research_packet(EvidenceStore(), first)
    store, second_stats = merge_research_packet(store, second)
    assert len(store.sources) == 1
    assert len(store.claims) == 1
    assert second_stats.reused_sources == 1
    assert second_stats.reused_claims == 1

    corroborating_payload = json.loads(json.dumps(base_draft))
    corroborating_url = "https://example.com/verification-study"
    corroborating_payload["sources"][0].update(
        {
            "title": "Independent Verification Evaluation",
            "url": corroborating_url,
            "source_type": "peer_reviewed",
        }
    )
    corroborating_payload["claims"][0]["source_urls"] = [corroborating_url]
    corroborating_payload["claims"][0]["confidence"] = "high"
    third = compile_research_packet(
        ResearchPacketDraft.model_validate(corroborating_payload),
        worker_id=3,
        retrieved_at="2026-01-01T00:02:00+00:00",
    )
    store, third_stats = merge_research_packet(store, third)
    assert len(store.sources) == 2
    assert len(store.claims) == 1
    assert len(next(iter(store.claims.values())).source_ids) == 2
    assert next(iter(store.claims.values())).confidence == "high"
    assert third_stats.reused_claims == 1

    brief = ResearchBrief(
        research_question="How do verification loops improve reliable tool use?",
        objective="Explain the mechanism and limitations.",
        scope_inclusions=["Tool execution verification"],
        scope_exclusions=[],
        constraints=["Use source-backed claims"],
        deliverable="A concise Markdown report.",
        success_criteria=["Cite the supporting source"],
    )
    context = select_writer_context(brief, store, max_claims=3)
    source_id = stable_id("src", canonicalize_url(source_url))
    claim_id = context.claims[0].id
    report = (
        f"# Verification Loops\n\n## Abstract\n\nVerification can detect failed tool "
        f"executions [{source_id}].\n\n## Limitations\n\nThe evidence is preliminary."
        f"\n\n## References\n\n- [{source_id}] Synthetic Verification Study — "
        f"{source_url}"
    )
    writer_output = EvidenceWriterOutput(
        markdown=report,
        used_claim_ids=[claim_id],
        used_source_ids=[source_id],
    )
    citation_result = validate_report_citations(writer_output, store)
    assert citation_result.valid is True

    provisional_source_id = "src_deadbeefcafe"
    provisional_output = EvidenceWriterOutput(
        markdown=report.replace(source_id, provisional_source_id),
        used_claim_ids=[claim_id],
        used_source_ids=[provisional_source_id],
    )
    canonical_output = canonicalize_writer_references(
        provisional_output, context, store
    )
    assert provisional_source_id not in canonical_output.markdown
    assert canonical_output.used_source_ids == [source_id]
    assert validate_report_citations(canonical_output, store).valid is True

    grouped_report = report.replace(
        f"[{source_id}].",
        f"[{source_id}, {provisional_source_id}].",
        1,
    )
    grouped_output = EvidenceWriterOutput(
        markdown=grouped_report,
        used_claim_ids=[claim_id],
        used_source_ids=[source_id, provisional_source_id],
    )
    grouped_validation = validate_report_citations(grouped_output, store)
    assert grouped_validation.valid is False
    assert provisional_source_id in grouped_validation.unknown_source_ids
    filtered_grouped_output = canonicalize_writer_references(
        grouped_output, context, store
    )
    assert provisional_source_id not in filtered_grouped_output.markdown
    assert validate_report_citations(filtered_grouped_output, store).valid is True

    result = {
        "canonical_source_deduplicated": True,
        "claim_deduplicated": True,
        "selected_claims": len(context.claims),
        "citation_validation": citation_result.valid,
        "provisional_id_remapped": True,
        "grouped_citation_validated": True,
        "first_merge": first_stats.model_dump(),
        "second_merge": second_stats.model_dump(),
        "corroborating_merge": third_stats.model_dump(),
    }
    print(f"V3 deterministic evidence check passed: {result}")
    return result


v3_regression_result = verify_v3_evidence_pipeline()

V3 deterministic evidence check passed: {'canonical_source_deduplicated': True, 'claim_deduplicated': True, 'selected_claims': 1, 'citation_validation': True, 'provisional_id_remapped': True, 'grouped_citation_validated': True, 'first_merge': {'added_sources': 1, 'reused_sources': 0, 'added_claims': 1, 'reused_claims': 0}, 'second_merge': {'added_sources': 0, 'reused_sources': 1, 'added_claims': 0, 'reused_claims': 1}, 'corroborating_merge': {'added_sources': 1, 'reused_sources': 0, 'added_claims': 0, 'reused_claims': 1}}


# 8. Optional Real End-to-End Run

The real run is disabled by default because it makes model and search calls. The deterministic
check above is safe to run whenever the Notebook restarts.

In [19]:
v3_demo_brief = ResearchBrief(
    research_question=(
        "What evidence supports reflection, tool verification, and multi-agent specialization "
        "as reliability patterns for LLM agents from 2023 onward?"
    ),
    objective=(
        "Help software engineers compare the mechanisms, evidence strength, and limitations "
        "of three reusable agent reliability patterns."
    ),
    scope_inclusions=[
        "Reflection or evaluator-optimizer loops",
        "Tool-call verification and recovery",
        "Multi-agent specialization and coordination",
    ],
    scope_exclusions=["General chat prompting without agent execution"],
    constraints=[
        "Focus on 2023 onward",
        "Prioritize primary academic and engineering sources",
        "Label arXiv-only evidence as preprint",
    ],
    deliverable="A source-backed Markdown report for software engineers.",
    success_criteria=[
        "Compare exactly three reliability patterns",
        "Map important claims to normalized sources",
        "Discuss evidence limitations",
    ],
)

RUN_V3_E2E = False

if RUN_V3_E2E:
    v3_result = await run_v3_evidence_research(
        research_brief=v3_demo_brief,
        max_tasks=3,
        max_parallel_workers=2,
        max_tool_rounds=4,
        max_writer_claims=16,
        run_semantic_evaluator=True,
        verbose=True,
    )
    show("V3 Evidence-Aware Report", v3_result.writer_output.markdown)

[EVIDENCE_PLANNER] Model call #1


[EVIDENCE_PLANNER] Planned tasks: 3
[EVIDENCE_WORKER:1] Starting isolated evidence task.
[EVIDENCE_WORKER:2] Starting isolated evidence task.


[EVIDENCE_WORKER:1] Tool call #1: arxiv_search_tool (items=5, errors=0)


[EVIDENCE_WORKER:2] Tool call #1: arxiv_search_tool (items=5, errors=0)


[EVIDENCE_WORKER:1] Tool call #2: arxiv_search_tool (items=5, errors=0)


...

[EVIDENCE_WORKER:3] Tool call #4: arxiv_search_tool (items=20, errors=0)


[EVIDENCE_WORKER:3] Completed: sources=4, claims=4, tool_calls=4
[V3_CONTROLLER] Evidence store: {'packets': 3, 'sources': 18, 'claims': 17, 'source_types': {'preprint': 18}, 'unresolved_questions': 17}
[V3_CONTROLLER] Reused sources: 0
[EVIDENCE_WRITER] Model call #1: claims=16, sources=16


[EVIDENCE_WRITER] Fail-closed citation filter: removed=['src_abbec7361004', 'src_c97e5b6c2cb3'], dropped_lines=1
[EVIDENCE_WRITER] References canonicalized: 15 cited sources


[V3_CONTROLLER] Report retained in notebook memory.


## V3 Evidence-Aware Report

## Abstract
This report evaluates three reusable reliability patterns for LLM agents, focusing on evidence published from 2023 onward. The patterns are: A) reflection/evaluator loops, B) tool-call verification and recovery, and C) multi-agent specialization and coordination. For each pattern, we map key claims to primary sources, summarize the strength of the evidence, and highlight notable limitations that affect real-world deployment for software engineers designing reliable agent systems. [src_a8d18a1d7dd0] [src_b299776a59ca][src_f45b71497b43][src_d49afb760843][src_141dc7e90074][src_7bb6350ef5fa][src_6ccd13beda25][src_121674e5f7d5][src_20971b530fb0][src_aaa67c4ce023][src_6efe267960bd][src_f58e8c61726e][src_12a63cae21eb][src_d49afb760843]

Note: This report uses a mixture of peer-reviewed, preprint, and benchmark-focused sources, with explicit labeling of arXiv preprints where applicable.


## Main analysis

### Pattern A: Reflection/evaluator loops
- Concept and goal: Reflection or evaluator loops aim to improve reliability by adding verification, self-assessment, or grounding checks inside multi-agent LLM workflows. The approach emphasizes timely verification rather than post hoc reconciliation to stabilize consensus among agents [src_a8d18a1d7dd0].
- Evidence and strength: Delaying verification in multi-agent LLM systems destabilizes consensus; the work derives instability thresholds and proposes optimal corrector placement to mitigate; grounding verification improves stability. This points to a nontrivial interaction between timing of verification and system stability, with grounding acting as a stabilizing mechanism [src_a8d18a1d7dd0]. Overall, the evidence is high confidence within the studied model/setup, but generalization to broader MA-LLM configurations remains uncertain [src_a8d18a1d7dd0].
- Related context and cautions: Theory-of-mind style coordination in multi-agent settings suggests ToM-enabled coordination can improve joint planning, but long-horizon reasoning and robust belief-state maintenance remain challenging components of reliable collaboration (see Pattern C discussion). This supports the notion that internal reasoning and verification need to be complemented by robust coordination strategies rather than relied on in isolation [src_aaa67c4ce023].
- Practical takeaway for engineers: If adopting reflection/evaluator loops, plan for careful placement of verifiers (correctors) and consider grounding the verification process to maintain system stability, especially in larger or more heterogeneous agent ensembles [src_a8d18a1d7dd0].

### Pattern B: Tool-call verification and recovery
- Concept and goal: Pattern B uses a modular tool-use architecture (planner, caller, summarizer) combined with stage-wise training to improve tool-use reliability and performance relative to monolithic LLM baselines. The architecture explicitly separates concerns to reduce failures when invoking external tools [src_abec7361004f].
- Evidence and strength: A modular multi-LLM agent architecture with planner, caller, and summarizer components and a two-stage training paradigm yields superior tool-use performance compared with traditional single-LLM baselines. This provides strong evidence that architectural separation and staged training improve tool-use reliability in benchmark tasks [src_abec7361004f].
- Tool quality and alignment: Semantics alone do not capture behavior-grounded utility; tool quality and alignment significantly affect reliability. Benchmarks with faulty tools reveal gaps that require robust tool-use policies and alignment considerations beyond surface semantic plausibility [src_b299776a59ca].
- Evaluation infrastructure: A deterministic evaluation framework with a tool sandbox (TREK) enables reproducible testing of LLM agents on complex, multi-constraint tasks with personas, supporting principled, reproducible assessment of tool-use reliability in controlled settings [src_141dc7e90074].
- Practical takeaway for engineers: Favor architectures that separate planning, tool invocation, and summarization, and invest in robust tool quality and alignment, along with reproducible evaluation benchmarks to validate reliability improvements before real-world deployment [src_abec7361004f][src_b299776a59ca][src_141dc7e90074].

### Pattern C: Multi-agent specialization and coordination
- Concept and goal: Pattern C centers on coordinated, specialized agents and ways to manage tool-usage privileges, coordination roles, and information fusion to improve reliability in group tasks.
- Evidence and strength: A set of studies supports the value of specialization and coordination strategies:
  - Over-privileged tool selection is common; privilege-aware post-training defenses can reduce unnecessary high-privilege tool use while preserving capabilities, contributing to safer and more reliable multi-agent behavior [src_7bb6350ef5fa].
  - Focus Agent demonstrates coordinated roles via simulated focus groups, illustrating practical coordination patterns for agent reliability in group tasks [src_6ccd13beda25].
  - LLM-based decentralized coordination shows robustness to unseen partners in pure coordination tasks, highlighting resilience in multi-agent configurations [src_121674e5f7d5].
  - Conventions to expand the action space (e.g., in Hanabi) yield significant improvements in self-/cross-play performance across team sizes, illustrating the practical gains from structured coordination conventions [src_20971b530fb0].
  - Theory of Mind for multi-agent collaboration via LLMs demonstrates emergent cooperative behaviors and ToM-enabled coordination improvements, though long-context maintenance remains a limitation for complex tasks [src_aaa67c4ce023].
  - MedAide explores information fusion and anatomy of medical intents via LLM-based agent collaboration, showing effective collaboration among specialized medical agents with dynamic role rotation and information fusion across tasks with composite intents [src_6efe267960bd].
- Practical takeaway for engineers: Coordination-centric patterns benefit from explicit role delineation, privilege-aware controls, and conventions that expand the action space. When deploying, ensure coordination protocols and world-model/state-tracking persist across agents to maintain reliability in group tasks. Cross-domain generalization should be validated, given domain-specific constraints in medical or risk-detection tasks [src_7bb6350ef5fa][src_6ccd13beda25][src_121674e5f7d5][src_20971b530fb0][src_aaa67c4ce023][src_6efe267960bd].

> Cross-pattern synthesis notes
>- Ensemble and multi-criteria evaluation: Across reliability phenomena and language tasks, no single metric dominates; ensembles and carefully designed metrics can yield more robust reliability assessments, underscoring the need for diverse evaluation approaches in agent reliability [src_f58e8c61726e].
>- Theoretical grounding and risk: Zero-shot analogical reasoning claims have been contested in the literature, reminding engineers to scrutinize extraordinary claims and demand robust empirical validation before extrapolating to production systems [src_daa6efe50b02].
>- Domain breadth vs generalization: Many findings come from domain-specific benchmarks (e.g., medical imaging, Hanabi, risk-detection tasks). Generalization to other domains or high-stakes deployments should be approached cautiously and with domain-adapted evaluation protocols [src_12a63cae21eb].

## Limitations
- Generalization uncertainty: Many results come from controlled benchmarks or domain-specific tasks; real-world deployment may encounter distribution shifts, unseen tools, or longer horizons that challenge the reported reliability gains [src_a8d18a1d7dd0][src_141dc7e90074][src_20971b530fb0].
- Preprint status and replication: Several influential sources are arXiv preprints or task-focused benchmarks; external replication and peer-reviewed confirmation are still progressing for some claims [src_aaa67c4ce023][src_f58e8c61726e].
- Tool realism and coverage: Tool-use reliability depends on tool quality and alignment; in real systems, tools may be faulty or unavailable, and failed tool calls can cascade, necessitating robust recovery policies and monitoring [src_b299776a59ca][src_abec7361004f][src_141dc7e90074].
- Long-horizon and context maintenance: For several coordination claims (ToM, focus groups, and multi-step plans), long-horizon reasoning and context maintenance remain challenging, which can limit reliability on complex, open-ended tasks [src_aaa67c4ce023].
- Evaluation scope: Some frameworks (e.g., TREK) provide deterministic benchmarks in synthetic or constrained knowledge bases; while valuable for reproducibility, these may not capture the full variability of production environments [src_141dc7e90074].

## References

- [src_121674e5f7d5] LLM-Coordination: Evaluating and Analyzing Multi-agent Coordination Abilities in Large Language Models — https://arxiv.org/abs/2310.03903v3
- [src_12a63cae21eb] Strategies to Harness the Transformers' Potential: UNSL at eRisk 2023 — http://arxiv.org/abs/2310.19970v1
- [src_141dc7e90074] TREK: A Travel Reasoning and Evaluation Kit for LLM Agents in Complex Trip Planning — http://arxiv.org/abs/2607.26977v2
- [src_20971b530fb0] Augmenting the action space with conventions to improve multi-agent cooperation in Hanabi — https://arxiv.org/abs/2412.06333v3
- [src_6ccd13beda25] Focus Agent: LLM-Powered Virtual Focus Group — http://arxiv.org/abs/2409.01907v1
- [src_6efe267960bd] MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration — https://arxiv.org/abs/2410.12532v3
- [src_7bb6350ef5fa] When Lower Privileges Suffice: Investigating Over-Privileged Tool Selection in LLM Agents — http://arxiv.org/abs/2606.20023v2
- [src_a8d18a1d7dd0] Delayed Verification Destabilizes Multi-Agent LLM Belief: Instability Thresholds and Optimal Corrector Placement — http://arxiv.org/abs/2606.27409v1
- [src_aaa67c4ce023] Theory of Mind for Multi-Agent Collaboration via Large Language Models — https://arxiv.org/abs/2310.10701v3
- [src_abec7361004f] Small LLMs Are Weak Tool Learners: A Multi-LLM Agent — http://arxiv.org/abs/2401.07324v3
- [src_b299776a59ca] RecoAtlas: From Semantic Plausibility to Set-Level Utility in LLM Recommendation Agents — http://arxiv.org/abs/2605.18805v1
- [src_d49afb760843] Overview of AuTexTification at IberLEF 2023: Detection and Attribution of Machine-Generated Text in Multiple Domains — http://arxiv.org/abs/2309.11285v1
- [src_daa6efe50b02] Response: Emergent analogical reasoning in large language models — http://arxiv.org/abs/2308.16118v2
- [src_f45b71497b43] Making Large Language Models Better Reasoners with Alignment — https://arxiv.org/abs/2309.02144v1
- [src_f58e8c61726e] ACES: Translation Accuracy Challenge Sets at WMT 2023 — http://arxiv.org/abs/2311.01153v1

# 9. Embedded V3 Report Artifact

The following report is the retained output of the successful V3 end-to-end run.

## Abstract
This report evaluates three reusable reliability patterns for LLM agents, focusing on evidence published from 2023 onward. The patterns are: A) reflection/evaluator loops, B) tool-call verification and recovery, and C) multi-agent specialization and coordination. For each pattern, we map key claims to primary sources, summarize the strength of the evidence, and highlight notable limitations that affect real-world deployment for software engineers designing reliable agent systems. [src_a8d18a1d7dd0] [src_b299776a59ca][src_f45b71497b43][src_d49afb760843][src_141dc7e90074][src_7bb6350ef5fa][src_6ccd13beda25][src_121674e5f7d5][src_20971b530fb0][src_aaa67c4ce023][src_6efe267960bd][src_f58e8c61726e][src_12a63cae21eb][src_d49afb760843]

Note: This report uses a mixture of peer-reviewed, preprint, and benchmark-focused sources, with explicit labeling of arXiv preprints where applicable.


## Main analysis

### Pattern A: Reflection/evaluator loops
- Concept and goal: Reflection or evaluator loops aim to improve reliability by adding verification, self-assessment, or grounding checks inside multi-agent LLM workflows. The approach emphasizes timely verification rather than post hoc reconciliation to stabilize consensus among agents [src_a8d18a1d7dd0].
- Evidence and strength: Delaying verification in multi-agent LLM systems destabilizes consensus; the work derives instability thresholds and proposes optimal corrector placement to mitigate; grounding verification improves stability. This points to a nontrivial interaction between timing of verification and system stability, with grounding acting as a stabilizing mechanism [src_a8d18a1d7dd0]. Overall, the evidence is high confidence within the studied model/setup, but generalization to broader MA-LLM configurations remains uncertain [src_a8d18a1d7dd0].
- Related context and cautions: Theory-of-mind style coordination in multi-agent settings suggests ToM-enabled coordination can improve joint planning, but long-horizon reasoning and robust belief-state maintenance remain challenging components of reliable collaboration (see Pattern C discussion). This supports the notion that internal reasoning and verification need to be complemented by robust coordination strategies rather than relied on in isolation [src_aaa67c4ce023].
- Practical takeaway for engineers: If adopting reflection/evaluator loops, plan for careful placement of verifiers (correctors) and consider grounding the verification process to maintain system stability, especially in larger or more heterogeneous agent ensembles [src_a8d18a1d7dd0].

### Pattern B: Tool-call verification and recovery
- Concept and goal: Pattern B uses a modular tool-use architecture (planner, caller, summarizer) combined with stage-wise training to improve tool-use reliability and performance relative to monolithic LLM baselines. The architecture explicitly separates concerns to reduce failures when invoking external tools [src_abec7361004f].
- Evidence and strength: A modular multi-LLM agent architecture with planner, caller, and summarizer components and a two-stage training paradigm yields superior tool-use performance compared with traditional single-LLM baselines. This provides strong evidence that architectural separation and staged training improve tool-use reliability in benchmark tasks [src_abec7361004f].
- Tool quality and alignment: Semantics alone do not capture behavior-grounded utility; tool quality and alignment significantly affect reliability. Benchmarks with faulty tools reveal gaps that require robust tool-use policies and alignment considerations beyond surface semantic plausibility [src_b299776a59ca].
- Evaluation infrastructure: A deterministic evaluation framework with a tool sandbox (TREK) enables reproducible testing of LLM agents on complex, multi-constraint tasks with personas, supporting principled, reproducible assessment of tool-use reliability in controlled settings [src_141dc7e90074].
- Practical takeaway for engineers: Favor architectures that separate planning, tool invocation, and summarization, and invest in robust tool quality and alignment, along with reproducible evaluation benchmarks to validate reliability improvements before real-world deployment [src_abec7361004f][src_b299776a59ca][src_141dc7e90074].

### Pattern C: Multi-agent specialization and coordination
- Concept and goal: Pattern C centers on coordinated, specialized agents and ways to manage tool-usage privileges, coordination roles, and information fusion to improve reliability in group tasks.
- Evidence and strength: A set of studies supports the value of specialization and coordination strategies:
  - Over-privileged tool selection is common; privilege-aware post-training defenses can reduce unnecessary high-privilege tool use while preserving capabilities, contributing to safer and more reliable multi-agent behavior [src_7bb6350ef5fa].
  - Focus Agent demonstrates coordinated roles via simulated focus groups, illustrating practical coordination patterns for agent reliability in group tasks [src_6ccd13beda25].
  - LLM-based decentralized coordination shows robustness to unseen partners in pure coordination tasks, highlighting resilience in multi-agent configurations [src_121674e5f7d5].
  - Conventions to expand the action space (e.g., in Hanabi) yield significant improvements in self-/cross-play performance across team sizes, illustrating the practical gains from structured coordination conventions [src_20971b530fb0].
  - Theory of Mind for multi-agent collaboration via LLMs demonstrates emergent cooperative behaviors and ToM-enabled coordination improvements, though long-context maintenance remains a limitation for complex tasks [src_aaa67c4ce023].
  - MedAide explores information fusion and anatomy of medical intents via LLM-based agent collaboration, showing effective collaboration among specialized medical agents with dynamic role rotation and information fusion across tasks with composite intents [src_6efe267960bd].
- Practical takeaway for engineers: Coordination-centric patterns benefit from explicit role delineation, privilege-aware controls, and conventions that expand the action space. When deploying, ensure coordination protocols and world-model/state-tracking persist across agents to maintain reliability in group tasks. Cross-domain generalization should be validated, given domain-specific constraints in medical or risk-detection tasks [src_7bb6350ef5fa][src_6ccd13beda25][src_121674e5f7d5][src_20971b530fb0][src_aaa67c4ce023][src_6efe267960bd].

> Cross-pattern synthesis notes
>- Ensemble and multi-criteria evaluation: Across reliability phenomena and language tasks, no single metric dominates; ensembles and carefully designed metrics can yield more robust reliability assessments, underscoring the need for diverse evaluation approaches in agent reliability [src_f58e8c61726e].
>- Theoretical grounding and risk: Zero-shot analogical reasoning claims have been contested in the literature, reminding engineers to scrutinize extraordinary claims and demand robust empirical validation before extrapolating to production systems [src_daa6efe50b02].
>- Domain breadth vs generalization: Many findings come from domain-specific benchmarks (e.g., medical imaging, Hanabi, risk-detection tasks). Generalization to other domains or high-stakes deployments should be approached cautiously and with domain-adapted evaluation protocols [src_12a63cae21eb].

## Limitations
- Generalization uncertainty: Many results come from controlled benchmarks or domain-specific tasks; real-world deployment may encounter distribution shifts, unseen tools, or longer horizons that challenge the reported reliability gains [src_a8d18a1d7dd0][src_141dc7e90074][src_20971b530fb0].
- Preprint status and replication: Several influential sources are arXiv preprints or task-focused benchmarks; external replication and peer-reviewed confirmation are still progressing for some claims [src_aaa67c4ce023][src_f58e8c61726e].
- Tool realism and coverage: Tool-use reliability depends on tool quality and alignment; in real systems, tools may be faulty or unavailable, and failed tool calls can cascade, necessitating robust recovery policies and monitoring [src_b299776a59ca][src_abec7361004f][src_141dc7e90074].
- Long-horizon and context maintenance: For several coordination claims (ToM, focus groups, and multi-step plans), long-horizon reasoning and context maintenance remain challenging, which can limit reliability on complex, open-ended tasks [src_aaa67c4ce023].
- Evaluation scope: Some frameworks (e.g., TREK) provide deterministic benchmarks in synthetic or constrained knowledge bases; while valuable for reproducibility, these may not capture the full variability of production environments [src_141dc7e90074].

## References

- [src_121674e5f7d5] LLM-Coordination: Evaluating and Analyzing Multi-agent Coordination Abilities in Large Language Models — https://arxiv.org/abs/2310.03903v3
- [src_12a63cae21eb] Strategies to Harness the Transformers' Potential: UNSL at eRisk 2023 — http://arxiv.org/abs/2310.19970v1
- [src_141dc7e90074] TREK: A Travel Reasoning and Evaluation Kit for LLM Agents in Complex Trip Planning — http://arxiv.org/abs/2607.26977v2
- [src_20971b530fb0] Augmenting the action space with conventions to improve multi-agent cooperation in Hanabi — https://arxiv.org/abs/2412.06333v3
- [src_6ccd13beda25] Focus Agent: LLM-Powered Virtual Focus Group — http://arxiv.org/abs/2409.01907v1
- [src_6efe267960bd] MedAide: Information Fusion and Anatomy of Medical Intents via LLM-based Agent Collaboration — https://arxiv.org/abs/2410.12532v3
- [src_7bb6350ef5fa] When Lower Privileges Suffice: Investigating Over-Privileged Tool Selection in LLM Agents — http://arxiv.org/abs/2606.20023v2
- [src_a8d18a1d7dd0] Delayed Verification Destabilizes Multi-Agent LLM Belief: Instability Thresholds and Optimal Corrector Placement — http://arxiv.org/abs/2606.27409v1
- [src_aaa67c4ce023] Theory of Mind for Multi-Agent Collaboration via Large Language Models — https://arxiv.org/abs/2310.10701v3
- [src_abec7361004f] Small LLMs Are Weak Tool Learners: A Multi-LLM Agent — http://arxiv.org/abs/2401.07324v3
- [src_b299776a59ca] RecoAtlas: From Semantic Plausibility to Set-Level Utility in LLM Recommendation Agents — http://arxiv.org/abs/2605.18805v1
- [src_d49afb760843] Overview of AuTexTification at IberLEF 2023: Detection and Attribution of Machine-Generated Text in Multiple Domains — http://arxiv.org/abs/2309.11285v1
- [src_daa6efe50b02] Response: Emergent analogical reasoning in large language models — http://arxiv.org/abs/2308.16118v2
- [src_f45b71497b43] Making Large Language Models Better Reasoners with Alignment — https://arxiv.org/abs/2309.02144v1
- [src_f58e8c61726e] ACES: Translation Accuracy Challenge Sets at WMT 2023 — http://arxiv.org/abs/2311.01153v1


# 10. V3 Completion Summary

V3 adds the evidence and context layer that V2 intentionally lacked:

- models extract semantic source-claim relationships;
- Python owns canonical URLs, stable IDs, deduplication, and referential integrity;
- isolated workers produce `ResearchPacketDraft` objects rather than free-form notes;
- `EvidenceStore` is the run-scoped source of truth;
- the Writer receives a bounded `WriterContext`;
- deterministic validation proves citation provenance;
- optional semantic evaluation checks whether citations actually support the prose.

V4 will use these observable structures to evaluate both final outcomes and execution
trajectories across a repeatable dataset.